In [1]:
from MyPlotRecipe.SharedX import ShareXaxis
from MyPlotRecipe.UniversalColor import UniversalColor
from MyPlotRecipe.legend_shadow import legend_shadow
import numpy as np
import math

from scipy import signal
import matplotlib.pyplot as plt
import matplotlib.ticker as ptick
from matplotlib.ticker import FixedLocator
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patheffects as pe

from CSField import CSField
import JupiterMag as jm

jm.Internal.Config(Model='jrm33', Degree=18, CartesianIn=True,
                   CartesianOut=True)
jm.Con2020.Config(equation_type='integral')

UC = UniversalColor()
UC.set_palette()

Importing Library
done


In [2]:
def day2hhmm(decimal_day):
    doy = int(decimal_day)
    rest = decimal_day - doy

    hours = int(rest*24)
    minutes = (rest*24-hours)*60

    if round(minutes) >= 60:
        hours += 1
        minutes = 0
    else:
        minutes = round(minutes)
    return hours, minutes


def hhmm2day(hours, minutes, seconds=0):
    fraction_of_day = hours/24 + minutes/(60*24) + seconds/(60*60*24)
    return fraction_of_day


def fmt_hhmm(decimal_day):
    hours, minutes = day2hhmm(decimal_day)
    hh = '{:02}'.format(hours)
    mm = '{:02}'.format(minutes)
    return hh+':'+mm

In [3]:
# Data from Connerney+2020: PJ index
con20_pj_idx = np.array([1, 3, 4, 5, 6,
                        7, 8, 9, 10, 11,
                        12, 13, 14, 15, 16,
                        17, 18, 19, 20, 21,
                        22, 23, 24], dtype=int)

# Data from Connerney+2020: Current constant [nT]
con20_mu_i_tot = np.array([150.1, 137.8, 127.2, 129.1, 130.1,
                           142.3, 140.1, 143.8, 137.0, 141.4,
                           124.2, 148.9, 145.3, 144.8, 149.9,
                           132.1, 133.5, 152.9, 138.5, 138.8,
                           156.1, 141.4, 146.3])

# Data from Connerney+2020: Radial current constant [MA]
con20_mu_i_rho = np.array([35.2, 14.6, 7.7, 11.5, 20.8,
                           20.2, 12.2, 21.1, 20.9, 10.7,
                           26.26, 16.4, 12.0, 19.6, 12.0,
                           13.6, 20.0, 12.8, 16.0, 17.3,
                           9.9, 16.1, 10.3])

In [4]:
# 定数
RJ = 71492*1E+3  # Jupiter radius [m]
RE = 1560*1E+3   # Europa radius [m]
# Angular velocity of Jupiter System III rotation [rad sec-1]
OMG_J = 2*np.pi/(9.55*3600)
OMG_G = 2*np.pi/(7.2*24*3600)   # Ganymede orbital angular velosity [rad/sec]

Psyn_eu = (11.22)*3600      # Moon's synodic period [sec]
Psyn_ga = (10.53)*3600      # Moon's synodic period [sec]

In [5]:
PJ_list = [1, 3, 4, 5,
           6, 7, 8, 9, 10,
           11, 12, 13, 14, 15,
           16, 17, 18, 19, 20,
           21, 22, 23, 24, 25,
           26, 27, 28, 29, 30,
           31, 32, 33, 34, 35,
           36, 37, 38, 39, 40,
           41, 42, 43, 44, 45,
           46, 47, 48, 49, 50,
           51, 52, 53, 54, 55,
           56, 57, 58, 59, 60,
           61, 62, 63, 64, 65,
           66, 67, 68, 69, 70,
           71, 72, 73, 74, 75,
           76, 77, 78, 79, 80]
dir = 'results/insitu_fit_a0_a1_b5_v2_36rj'
f0 = np.loadtxt(dir+'/result_PJ' +
                str(PJ_list[0]).zfill(2)+'_PJ'+str(PJ_list[-1]).zfill(2)+'.txt')

In [6]:
# ===========================================================
# CONFIGURE CSFIELD
# ===========================================================
csfield = CSField()
csfield.config(
    i_phi0=4.838275473116932E-5,  # 4.845093370096503E-5
    I_rho0=10.033388761574646,    # 10.45591036962724
    a0=2.8333677E-6,              # 1.87067601E-6
    MLT0=1.56400572,              # 2.7409048
    a1=14.95621835,               # 25.56805279
    MLT1=19.83627108,             # 18.83227378
    b1=4.2518,
    MLTb=-0.2086,
    D=2.5*math.sqrt(5/3),          # 2.5
    # c1=5.0,         # 5.0
    # c2=14.0,        # 14.0
    # c3=46.0,        # 46.0
    # w1=-0.06,         # -0.06
    # w2=1.39,          # 1.39
    # w3=-0.82,         # -0.82
)

In [7]:
con2020_total_rmse = np.zeros(con20_pj_idx.size)

# PJ6のデータを読み込む

In [8]:
for j in range(con20_pj_idx.size):
    PJ_num = con20_pj_idx[j]

    dir = 'data/FGM/PJ'+str(PJ_num).zfill(2)+'/'
    time_arr = np.loadtxt(
        dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_time.txt')
    Bx_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_Bx.txt')
    By_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_By.txt')
    Bz_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_Bz.txt')
    rx_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_rx.txt')
    ry_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_ry.txt')
    rz_pc = np.loadtxt(dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_rz.txt')
    JunoMLT = np.loadtxt(
        dir+'fgm_PJ'+str(PJ_num).zfill(2)+'_pc_2min_JunoMLT.txt')

    rx_pc *= 1/71492.0  # [RJ]
    ry_pc *= 1/71492.0  # [RJ]
    rz_pc *= 1/71492.0  # [RJ]

    # 座標
    r_pc = np.sqrt(rx_pc**2+ry_pc**2+rz_pc**2)
    cos_phi = rx_pc/r_pc
    sin_phi = ry_pc/r_pc
    cos_theta = rz_pc/r_pc
    sin_theta = np.sqrt(rx_pc**2+ry_pc**2)/r_pc

    # ===========================================================
    # Internal field
    # ===========================================================
    Bx0_pc = np.zeros(time_arr.size)
    By0_pc = np.zeros(time_arr.size)
    Bz0_pc = np.zeros(time_arr.size)
    for i in range(time_arr.size):
        Bx0_pc[i], By0_pc[i], Bz0_pc[i] = jm.Internal.Field(rx_pc[i],
                                                            ry_pc[i],
                                                            rz_pc[i])  # [nT]
    dBx = Bx_pc - Bx0_pc
    dBy = By_pc - By0_pc
    dBz = Bz_pc - Bz0_pc
    dBr = dBx*sin_theta*cos_phi + dBy*sin_theta*sin_phi + dBz*cos_theta
    dBtheta = dBx*cos_theta*cos_phi + dBy*cos_theta*sin_phi - dBz*sin_theta
    dBphi = -dBx*sin_phi + dBy*cos_phi

    # ===========================================================
    # Con2020 original value
    # ===========================================================
    jm.Con2020.Config(
        mu_i=con20_mu_i_tot[np.where(con20_pj_idx == PJ_num)],
        i_rho=con20_mu_i_rho[np.where(con20_pj_idx == PJ_num)],
    )
    dBx1_pc = np.zeros(time_arr.size)
    dBy1_pc = np.zeros(time_arr.size)
    dBz1_pc = np.zeros(time_arr.size)
    for i in range(time_arr.size):
        dBx1_pc[i], dBy1_pc[i], dBz1_pc[i] = jm.Con2020.Field(rx_pc[i],
                                                              ry_pc[i],
                                                              rz_pc[i])  # [nT]
    dBr1_pc = dBx1_pc*sin_theta*cos_phi + dBy1_pc * \
        sin_theta*sin_phi + dBz1_pc*cos_theta
    dBtheta1_pc = dBx1_pc*cos_theta*cos_phi + \
        dBy1_pc*cos_theta*sin_phi - dBz1_pc*sin_theta
    dBphi1_pc = -dBx1_pc*sin_phi + dBy1_pc*cos_phi

    # ===========================================================
    # RMSE（Root Mean Squared Error）
    # ===========================================================
    x_cs, y_cs, _ = csfield._sys3_2_cs(rx_pc, ry_pc, rz_pc)
    phi_cs = np.arctan2(y_cs, x_cs)
    rho_cs = np.sqrt(x_cs**2 + y_cs**2)
    in_30rj = np.where(r_pc < 36.0)[0][0]
    out_30rj = np.where(r_pc < 36.0)[0][-1]
    if PJ_num in [17]:
        in_5rj = np.where(rho_cs < 8.0)[0][0]
        out_5rj = np.where(rho_cs < 8.0)[0][-1]
    else:
        in_5rj = np.where(rho_cs < 5.0)[0][0]
        out_5rj = np.where(rho_cs < 5.0)[0][-1]
    in_num = in_5rj-in_30rj-1
    out_num = out_30rj-out_5rj-1

    # ===========================================================
    # Total RMSE
    # ===========================================================
    RMS_total = math.sqrt(
        (
            np.sum(
                (dBr[in_30rj:in_5rj]-dBr1_pc[in_30rj:in_5rj])**2
                + (dBtheta[in_30rj:in_5rj]-dBtheta1_pc[in_30rj:in_5rj])**2
                + (dBphi[in_30rj:in_5rj]-dBphi1_pc[in_30rj:in_5rj])**2
            ) + np.sum(
                (dBr[out_5rj:out_30rj]-dBr1_pc[out_5rj:out_30rj])**2
                + (dBtheta[out_5rj:out_30rj]-dBtheta1_pc[out_5rj:out_30rj])**2
                # +(dBphi[out_5rj:out_30rj]-dBphi1_pc[out_5rj:out_30rj])**2
            )
        )/(3*in_num+2*out_num))
    print('PJ'+str(PJ_num).zfill(2)+' RMS total:', RMS_total)

    con2020_total_rmse[j] = RMS_total

PJ01 RMS total: 3.76647032308328
PJ03 RMS total: 2.7539837253971213
PJ04 RMS total: 2.4261899364910198
PJ05 RMS total: 2.8731292720593546
PJ06 RMS total: 2.8692724884074945
PJ07 RMS total: 3.7064294103736612
PJ08 RMS total: 3.5964744264319646
PJ09 RMS total: 3.3737483018896457
PJ10 RMS total: 3.3227933646088395
PJ11 RMS total: 3.1855423313934415
PJ12 RMS total: 3.4843193542151125
PJ13 RMS total: 3.9991325001063034
PJ14 RMS total: 3.161710710876049
PJ15 RMS total: 3.3847280875923547
PJ16 RMS total: 3.5999157737551934
PJ17 RMS total: 2.971798953490263
PJ18 RMS total: 2.867089811457003
PJ19 RMS total: 3.8852646959967494
PJ20 RMS total: 3.0097715669615264
PJ21 RMS total: 3.422226770741107
PJ22 RMS total: 5.115202269826201
PJ23 RMS total: 3.3940034150331773
PJ24 RMS total: 4.660315271450619


In [9]:
np.savetxt('results/con2020_total_rmse.txt',
           con2020_total_rmse)